# Notebook 3 — Revised Experiments: Robust, Stability-Aware Multi-Objective Feature Selection (CICIoMT2024)

This is the **revised** experiment. It replaces the two runs of Notebook 2 with **one** run.

### What changed compared with Notebook 2, and why

| Change | Why |
|---|---|
| **IAT is excluded from all feature selection.** One extra reference row ("All features + IAT") still shows how much IAT inflates the scores. | IAT behaves like a capture timestamp (leakage). One run now covers both stories. |
| **Realistic noise test.** Only real-valued measurements (sizes, rates, times, counts) get noise, as a ±% measurement error. 0/1 indicator features are left unchanged. Three noise levels (5%, 10%, 20%), each averaged over 2 random draws. | The old test pushed 0/1 flags to impossible values, which is unrealistic and harsh. |
| **More reliable optimiser scoring.** Each candidate subset is scored with 2-fold cross-validation inside the training data, and the search is larger (population 24, 20 generations). | Less random scoring means less random feature choices. |
| **Stability-aware consensus selection (proposed).** Instead of taking one solution, the final subset keeps the features chosen by **at least half** of all near-optimal subsets found during the search (near-optimal in both accuracy and robustness). | Picking one solution made the selected features change a lot from run to run. |
| **Class-balanced Random Forest** (`class_weight="balanced_subsample"`) for every method. | Helps rare attacks (ARP spoofing, Recon-VulScan) that were often missed. |
| **Memory-safe and crash-safe.** Progress saved after every run; results saved **before** figures; model size estimated without copying the model. | Prevents the crashes seen before. |

### Ablation ladder (what each component adds)

| Method | Error | Size | Robustness objective | Consensus |
|---|---|---|---|---|
| NSGA-II (2-obj) | ✓ | ✓ | | |
| NSGA-II (3-obj) | ✓ | ✓ | ✓ | |
| **Proposed: NSGA-II (3-obj) + consensus** | ✓ | ✓ | ✓ | ✓ |

**How to run:** put this notebook in the same folder as `cleaned_data` (made by Notebook 1). Check that the kernel (top right) is **Python (iomt)**, then choose **Kernel → Restart Kernel and Run All Cells**. If it is interrupted, choose Run All again: it continues where it stopped.

## Step 1 — Tools and settings

In [ ]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # prevents a common Windows crash (duplicate OpenMP library)
import sys, subprocess
try:
    import pymoo
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pymoo"])

In [ ]:
import gc, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             average_precision_score, confusion_matrix)
from sklearn.preprocessing import label_binarize
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.operators.crossover.pntx import TwoPointCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from pymoo.optimize import minimize
warnings.filterwarnings("ignore")

# ------------------------- SETTINGS -------------------------
CLEAN_DIR = "cleaned_data"      # output folder of Notebook 1
OUT       = "results_revised"
N_RUNS    = 10                  # independent repetitions (different random seeds)
TRAIN_PER_CLASS = 20000         # training rows sampled per class in each run
TEST_PER_CLASS  = 20000         # test rows per class (same test set for every run)
NOISE_LEVELS = [0.05, 0.10, 0.20]   # relative measurement error on real-valued features
NOISE_MAIN   = 0.10             # level used inside the optimiser and in the main table
NOISE_DRAWS  = 2                # random noise draws per level (results are averaged)
GA_POP, GA_GEN = 24, 20         # NSGA-II population size and generations
GA_PER_CLASS = 800              # rows per class used inside the optimiser
GA_FOLDS     = 2                # cross-validation folds used to score each candidate
GA_TREES     = 10               # trees in the small forest used inside the optimiser
F1_TOLERANCE = 0.01             # "near-optimal" = validation F1 within 0.01 of the best
CONSENSUS    = 0.5              # keep features chosen by >= 50% of near-optimal solutions
FINAL_TREES  = 100              # trees in the final Random Forest
# -------------------------------------------------------------

os.makedirs(OUT, exist_ok=True)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 10,
                     "font.family": "serif", "axes.grid": True, "grid.alpha": 0.3})
print("Settings ready. Results folder:", os.path.abspath(OUT))

## Step 2 — Read the cleaned data (from Notebook 1)

In [ ]:
train_all = pd.read_csv(os.path.join(CLEAN_DIR, "train_clean.csv"))
test_all  = pd.read_csv(os.path.join(CLEAN_DIR, "test_clean.csv"))
FEATURES_ALL = [c for c in train_all.columns if c != "Label"]
train_all[FEATURES_ALL] = train_all[FEATURES_ALL].astype("float32")
test_all[FEATURES_ALL]  = test_all[FEATURES_ALL].astype("float32")
CLASSES = sorted(set(train_all["Label"]) & set(test_all["Label"]))

# Features used for selection: everything except the timestamp-like IAT
FEATURES = [c for c in FEATURES_ALL if c != "IAT"]
IAT_IN_DATA = "IAT" in FEATURES_ALL

# 0/1 indicator features (flags, protocol indicators): values always between 0 and 1 in training
INDICATOR = np.array([train_all[c].min() >= 0 and train_all[c].max() <= 1 for c in FEATURES_ALL])
print("Train rows:", f"{len(train_all):,}", "| Test rows:", f"{len(test_all):,}")
print("Features for selection (IAT excluded):", len(FEATURES))
print("Indicator (0-1) features, never perturbed:", int(INDICATOR.sum()),
      "| real-valued features, perturbed in the noise test:", int((~INDICATOR).sum()))

## Step 3 — The classes

In [ ]:
def category(lbl):
    if lbl == "Benign": return "Benign"
    if "Spoofing" in lbl: return "Spoofing"
    if lbl.startswith("MQTT"):  return "MQTT"
    if lbl.startswith("Recon"): return "Recon"
    if "DDoS" in lbl: return "DDoS"
    if "DoS" in lbl:  return "DoS"
    return "Other"

class_table = pd.DataFrame({
    "Category": [category(c) for c in CLASSES],
    "Train rows (clean)": train_all["Label"].value_counts().reindex(CLASSES).values,
    "Test rows (clean)":  test_all["Label"].value_counts().reindex(CLASSES).values,
}, index=CLASSES)
class_table["Train rows used per run"] = np.minimum(class_table["Train rows (clean)"], TRAIN_PER_CLASS)
class_table["Test rows used"] = np.minimum(class_table["Test rows (clean)"], TEST_PER_CLASS)
class_table

## Step 4 — Fixed test set and realistic noisy copies

**Noise model:** each real-valued measurement *x* becomes *x × (1 + ε)*, where ε is drawn from a normal distribution with standard deviation 5%, 10% or 20%, and the result is kept ≥ 0. Indicator features are not changed. The same noisy copies are used for every method, so the comparison is fair.

In [ ]:
def sample_per_class(df, label_col, n, seed):
    # take up to n random rows from each class (works on old and new pandas)
    idx = [g.sample(min(len(g), n), random_state=seed).index for _, g in df.groupby(label_col)]
    return df.loc[np.concatenate(idx)]

def perturb(X, level, rng, indicator):
    # relative measurement error on real-valued columns only; indicators untouched
    Xn = X.copy()
    cont = ~indicator
    Xn[:, cont] = np.clip(X[:, cont] * (1 + rng.normal(0, level, (X.shape[0], cont.sum()))), 0, None)
    return Xn.astype("float32")

test_set = sample_per_class(test_all, "Label", TEST_PER_CLASS, seed=42)
X_test_full = test_set[FEATURES_ALL].values          # all columns, incl. IAT
y_test = test_set["Label"].values
COL = {f: i for i, f in enumerate(FEATURES_ALL)}
SEL_COLS = np.array([COL[f] for f in FEATURES])      # column positions of the selectable features

X_test_noisy = {lvl: [perturb(X_test_full, lvl, np.random.default_rng(1000 * d + int(lvl * 100)), INDICATOR)
                      for d in range(NOISE_DRAWS)] for lvl in NOISE_LEVELS}
del test_all, test_set; gc.collect()                  # free memory

def sample_train(seed):
    s = sample_per_class(train_all, "Label", TRAIN_PER_CLASS, seed)
    return s[FEATURES_ALL].values, s["Label"].values

print("Test set:", X_test_full.shape, "| noisy copies:", len(NOISE_LEVELS) * NOISE_DRAWS)

## Step 5 — How every feature subset is trained and scored

Same classifier for every method: a **class-balanced Random Forest** with 100 trees.

| Metric | Meaning |
|---|---|
| Macro-F1, precision, recall, PR-AUC | detection quality, every class weighted equally |
| FPR / FNR | benign traffic flagged as attack / attacks called benign |
| F1 at 5%, 10%, 20% noise | detection quality under measurement error (mean of 2 draws) |
| F1 drop (10%) | clean F1 minus F1 at 10% noise (lower = more robust) |
| Train time, inference time | seconds to train; ms to classify 1,000 records |
| Model size (MB, est.) | estimated memory of the forest (from its number of tree nodes) |

In [ ]:
def make_forest(n_trees, seed):
    return RandomForestClassifier(n_estimators=n_trees, class_weight="balanced_subsample",
                                  random_state=seed, n_jobs=-1)

def evaluate(cols, X_tr, y_tr, seed):
    cols = list(cols)
    model = make_forest(FINAL_TREES, seed)
    t0 = time.perf_counter(); model.fit(X_tr[:, cols], y_tr); train_time = time.perf_counter() - t0
    t0 = time.perf_counter(); pred = model.predict(X_test_full[:, cols]); infer = time.perf_counter() - t0
    proba = model.predict_proba(X_test_full[:, cols])

    f1 = f1_score(y_test, pred, average="macro")
    benign = y_test == "Benign"
    res = {"n_features": len(cols), "macro_F1": f1,
           "precision": precision_score(y_test, pred, average="macro", zero_division=0),
           "recall": recall_score(y_test, pred, average="macro", zero_division=0),
           "PR_AUC": average_precision_score(label_binarize(y_test, classes=model.classes_), proba, average="macro"),
           "FPR": np.mean(pred[benign] != "Benign"), "FNR": np.mean(pred[~benign] == "Benign")}
    for lvl in NOISE_LEVELS:
        res[f"F1_noise_{int(lvl*100)}"] = np.mean([f1_score(y_test, model.predict(Xn[:, cols]), average="macro")
                                                   for Xn in X_test_noisy[lvl]])
    res["F1_drop_10"] = f1 - res["F1_noise_10"]
    nodes = sum(e.tree_.node_count for e in model.estimators_)
    res.update({"train_time_s": train_time,
                "infer_ms_per_1000": 1000 * infer / len(X_test_full) * 1000,
                "model_MB_est": nodes * (64 + 8 * len(CLASSES)) / 1e6})
    per_class = dict(zip(CLASSES, f1_score(y_test, pred, average=None, labels=CLASSES)))
    del model; gc.collect()
    return res, per_class, pred

## Step 6 — Feature-selection methods

| Method | How it chooses features |
|---|---|
| **All features + IAT (reference)** | everything, including IAT: shows how much the timestamp-like feature inflates scores |
| **All features** | everything except IAT (no selection) |
| **MI (top-k)** | mutual information ranking, keep top *k* |
| **RFE (top-k)** | recursive feature elimination with a Random Forest, keep *k* |
| **NSGA-II (2-obj)** | minimise error and number of features; pick one solution |
| **NSGA-II (3-obj)** | adds the robustness objective (F1 drop under 10% noise); pick one solution |
| **Proposed** | NSGA-II (3-obj) + **consensus**: among **all** subsets evaluated during the search, take those within 0.01 of the best cross-validated F1 **and** within 0.01 of the smallest noise drop; keep the features used by ≥ 50% of them |

*k* for MI and RFE equals the number of features of the proposed method in that run (fair comparison).

Each candidate inside NSGA-II is scored with **2-fold cross-validation** on a small balanced part of the **training** data only. The test set is never used for selection.

In [ ]:
class FeatureSelectionProblem(ElementwiseProblem):
    def __init__(self, folds, n_obj, seed):
        super().__init__(n_var=len(SEL_COLS), n_obj=n_obj, xl=0, xu=1, vtype=bool)
        self.folds, self.seed = folds, seed
        self.archive = []                       # every candidate evaluated during the search

    def _evaluate(self, x, out, *args, **kwargs):
        idx = np.where(x)[0]
        if len(idx) == 0:
            out["F"] = [1.0, 0.0, 1.0][: self.n_obj]; return
        f1s, drops = [], []
        for Xa, ya, Xv, yv, Xvn in self.folds:
            clf = make_forest(GA_TREES, self.seed).fit(Xa[:, idx], ya)
            f1 = f1_score(yv, clf.predict(Xv[:, idx]), average="macro")
            f1s.append(f1)
            if self.n_obj == 3:
                drops.append(max(0.0, f1 - f1_score(yv, clf.predict(Xvn[:, idx]), average="macro")))
        F = [1 - np.mean(f1s), len(idx) / self.n_var]
        if self.n_obj == 3:
            F.append(np.mean(drops))
        out["F"] = F
        self.archive.append((np.asarray(x, bool).copy(), np.array(F)))

def make_folds(X_tr, y_tr, seed):
    df = pd.DataFrame(X_tr[:, SEL_COLS]); df["y"] = y_tr
    sub = sample_per_class(df, "y", GA_PER_CLASS, seed)
    Xs, ys = sub.drop(columns="y").values.astype("float32"), sub["y"].values
    ind = INDICATOR[SEL_COLS]
    folds = []
    for f, (a, v) in enumerate(StratifiedKFold(GA_FOLDS, shuffle=True, random_state=seed).split(Xs, ys)):
        Xvn = perturb(Xs[v], NOISE_MAIN, np.random.default_rng(seed * 10 + f), ind)
        folds.append((Xs[a], ys[a], Xs[v], ys[v], Xvn))
    return folds

def nsga2(folds, seed, n_obj):
    algo = NSGA2(pop_size=GA_POP, sampling=BinaryRandomSampling(), crossover=TwoPointCrossover(),
                 mutation=BitflipMutation(), eliminate_duplicates=True)
    problem = FeatureSelectionProblem(folds, n_obj, seed)
    res = minimize(problem, algo, ("n_gen", GA_GEN), seed=seed, verbose=False)
    archX = np.array([a[0] for a in problem.archive]); archF = np.array([a[1] for a in problem.archive])
    return np.atleast_2d(res.F), np.atleast_2d(res.X), archF, archX

def pick_single(F, X, n_obj):
    # Pareto front: among near-optimal solutions take the fewest features (ties: smallest noise drop)
    val_f1 = 1 - F[:, 0]
    ok = np.where(val_f1 >= val_f1.max() - F1_TOLERANCE)[0]
    tie = F[ok, 2] if n_obj == 3 else np.zeros(len(ok))
    best = ok[np.lexsort((tie, F[ok, 1]))[0]]
    return SEL_COLS[np.where(X[best])[0]]

def pick_consensus(archF, archX):
    # all candidates evaluated during the search that are near-optimal in BOTH accuracy and robustness
    val_f1, drop = 1 - archF[:, 0], archF[:, 2]
    good = val_f1 >= val_f1.max() - F1_TOLERANCE
    good &= drop <= drop[good].min() + F1_TOLERANCE
    near = np.unique(archX[good], axis=0)          # count each distinct subset once
    chosen = np.where(near.mean(axis=0) >= CONSENSUS)[0]
    if len(chosen) == 0:
        chosen = np.where(near[0])[0]
    return SEL_COLS[chosen], len(near)

def mi_select(X_tr, y_tr, k, seed):
    idx = np.random.default_rng(seed).choice(len(X_tr), min(20000, len(X_tr)), replace=False)
    scores = mutual_info_classif(X_tr[idx][:, SEL_COLS], y_tr[idx], random_state=seed)
    return SEL_COLS[np.argsort(scores)[::-1][:k]]

def rfe_select(X_tr, y_tr, k, seed):
    idx = np.random.default_rng(seed).choice(len(X_tr), min(10000, len(X_tr)), replace=False)
    rfe = RFE(make_forest(30, seed), n_features_to_select=k, step=2).fit(X_tr[idx][:, SEL_COLS], y_tr[idx])
    return SEL_COLS[np.where(rfe.support_)[0]]

ALL_WITH_IAT = np.arange(len(FEATURES_ALL))
PROPOSED = "Proposed: NSGA-II (3-obj) + consensus"
print("Methods defined.")

## Step 7 — Run the experiments

Progress is printed after every run and **saved** to `checkpoint.pkl`. If anything interrupts the notebook, choose **Run All** again: finished runs are loaded and it continues.

In [ ]:
CHECKPOINT = os.path.join(OUT, "checkpoint.pkl")
signature = (TRAIN_PER_CLASS, TEST_PER_CLASS, tuple(NOISE_LEVELS), NOISE_MAIN, NOISE_DRAWS, GA_POP, GA_GEN,
             GA_PER_CLASS, GA_FOLDS, GA_TREES, F1_TOLERANCE, CONSENSUS, FINAL_TREES,
             tuple(FEATURES_ALL), tuple(CLASSES))

rows, per_class_rows, selected = [], [], []
pareto_example, preds_example, first_run = None, None, 0
if os.path.exists(CHECKPOINT):
    with open(CHECKPOINT, "rb") as fh:
        ck = pickle.load(fh)
    if ck["signature"] == signature:
        rows, per_class_rows, selected = ck["rows"], ck["per_class_rows"], ck["selected"]
        pareto_example, preds_example, first_run = ck["pareto_example"], ck["preds_example"], ck["next_run"]
        print(f"Saved progress found: {first_run} run(s) already finished - continuing from run {first_run + 1}.")
    else:
        print("Settings changed since the saved progress - starting fresh.")

t_start = time.time()
for run in range(first_run, N_RUNS):
    seed = run
    X_tr, y_tr = sample_train(seed)
    folds = make_folds(X_tr, y_tr, seed)

    t0 = time.perf_counter(); F2, X2, _, _ = nsga2(folds, seed, n_obj=2); t2 = time.perf_counter() - t0
    t0 = time.perf_counter(); F3, X3, pF3, pX3 = nsga2(folds, seed, n_obj=3); t3 = time.perf_counter() - t0
    sel2 = pick_single(F2, X2, 2)
    sel3 = pick_single(F3, X3, 3)
    selP, n_near = pick_consensus(pF3, pX3)
    k = len(selP)
    t0 = time.perf_counter(); sel_mi  = mi_select(X_tr, y_tr, k, seed);  t_mi  = time.perf_counter() - t0
    t0 = time.perf_counter(); sel_rfe = rfe_select(X_tr, y_tr, k, seed); t_rfe = time.perf_counter() - t0

    methods = {"All features": (SEL_COLS, 0.0), "MI (top-k)": (sel_mi, t_mi), "RFE (top-k)": (sel_rfe, t_rfe),
               "NSGA-II (2-obj)": (sel2, t2), "NSGA-II (3-obj)": (sel3, t3), PROPOSED: (selP, t3)}
    if IAT_IN_DATA:
        methods = {"All features + IAT (reference)": (ALL_WITH_IAT, 0.0), **methods}
    for name, (cols, sel_time) in methods.items():
        res, pc, pred = evaluate(cols, X_tr, y_tr, seed)
        rows.append({"run": run, "method": name, "selection_time_s": sel_time, **res})
        per_class_rows.append({"run": run, "method": name, **pc})
        selected.append({"run": run, "method": name, "features": [FEATURES_ALL[i] for i in cols]})
        if run == 0 and name == PROPOSED:
            pareto_example, preds_example = F3, pred

    with open(CHECKPOINT, "wb") as fh:
        pickle.dump({"signature": signature, "rows": rows, "per_class_rows": per_class_rows, "selected": selected,
                     "pareto_example": pareto_example, "preds_example": preds_example, "next_run": run + 1}, fh)
    del X_tr, y_tr, folds; gc.collect()
    print(f"Run {run+1}/{N_RUNS} done | proposed k = {k} (consensus of {n_near} near-optimal subsets) | "
          f"elapsed {(time.time()-t_start)/60:.1f} min | progress saved")

results = pd.DataFrame(rows)
per_class = pd.DataFrame(per_class_rows)
selected = pd.DataFrame(selected)
print("\nAll runs finished.")

## Step 8 — Main results table (mean ± 95% confidence interval over runs)

In [ ]:
ORDER = ["All features + IAT (reference)", "All features", "MI (top-k)", "RFE (top-k)",
         "NSGA-II (2-obj)", "NSGA-II (3-obj)", PROPOSED]
METHOD_ORDER = [m for m in ORDER if m in set(results["method"])]
NOISE_COLS = [f"F1_noise_{int(l*100)}" for l in NOISE_LEVELS]
METRICS = ["n_features", "macro_F1", "precision", "recall", "PR_AUC", "FPR", "FNR"] + NOISE_COLS + \
          ["F1_drop_10", "selection_time_s", "train_time_s", "infer_ms_per_1000", "model_MB_est"]

def ci95(x):
    x = np.asarray(x, float)
    return 0.0 if len(x) < 2 else stats.t.ppf(0.975, len(x) - 1) * x.std(ddof=1) / np.sqrt(len(x))

mean_t = results.groupby("method")[METRICS].mean().reindex(METHOD_ORDER)
ci_t   = results.groupby("method")[METRICS].agg(ci95).reindex(METHOD_ORDER)
summary = mean_t.copy().astype(object)
for m in METRICS:
    dec = 2 if m in ("n_features", "selection_time_s", "train_time_s", "infer_ms_per_1000", "model_MB_est") else 4
    summary[m] = [f"{a:.{dec}f} ± {b:.{dec}f}" for a, b in zip(mean_t[m], ci_t[m])]
summary.T

## Step 9 — Statistical tests

Friedman test across all methods, then Wilcoxon signed-rank tests (paired by run) comparing the proposed method with each other method, with Holm correction.

In [ ]:
def holm(pvals):
    p = np.asarray(pvals); order = np.argsort(p); adj = np.empty_like(p); m = len(p); running = 0
    for rank, i in enumerate(order):
        running = max(running, min(1, (m - rank) * p[i])); adj[i] = running
    return adj

stat_rows = []
for metric in ["macro_F1", "F1_noise_10", "F1_drop_10", "n_features", "infer_ms_per_1000", "train_time_s"]:
    wide = results.pivot(index="run", columns="method", values=metric)[METHOD_ORDER]
    fr = stats.friedmanchisquare(*[wide[c] for c in METHOD_ORDER])
    prop = wide[PROPOSED]; comps = []
    for other in [m for m in METHOD_ORDER if m != PROPOSED]:
        d = prop - wide[other]
        p = 1.0 if np.allclose(d, 0) else stats.wilcoxon(prop, wide[other]).pvalue
        comps.append((other, d.mean(), p))
    for (other, diff, p), pa in zip(comps, holm([c[2] for c in comps])):
        stat_rows.append({"metric": metric, "Friedman p": fr.pvalue, "proposed vs": other,
                          "mean difference (proposed - other)": diff, "Wilcoxon p": p,
                          "Holm-adjusted p": pa, "significant (0.05)": "yes" if pa < 0.05 else "no"})
stats_table = pd.DataFrame(stat_rows)
stats_table.round(5)

## Step 10 — Stability of the selected features (Jaccard index between runs)

In [ ]:
def pairwise_jaccard(sets):
    return [len(a & b) / len(a | b) for i, a in enumerate(sets) for b in sets[i + 1:] if a | b]

stab = []
for m in ["MI (top-k)", "RFE (top-k)", "NSGA-II (2-obj)", "NSGA-II (3-obj)", PROPOSED]:
    sets = [set(f) for f in selected[selected["method"] == m]["features"]]
    j = pairwise_jaccard(sets)
    stab.append({"method": m, "mean Jaccard stability": np.mean(j), "std": np.std(j),
                 "mean subset size": np.mean([len(s) for s in sets])})
stability = pd.DataFrame(stab)

prop_sets = selected[selected["method"] == PROPOSED]["features"]
freq = pd.Series([f for s in prop_sets for f in s]).value_counts().reindex(FEATURES, fill_value=0)
freq = (freq / results["run"].nunique()).sort_values(ascending=False).rename("selection frequency")
display(stability.round(4))
freq.head(20)

## Step 11 — Results per class (mean F1 over runs)

In [ ]:
per_class_mean = per_class.groupby("method")[CLASSES].mean().reindex(METHOD_ORDER).T
per_class_mean.insert(0, "Test rows", class_table["Test rows used"])
per_class_mean.round(4)

## Step 12 — Save everything (before figures)

- `results_revised/experiment_results.xlsx` – every table
- `results_revised/summary_for_manuscript.txt` – **send this back**

In [ ]:
cm_run1 = confusion_matrix(y_test, preds_example, labels=CLASSES, normalize="true")
with pd.ExcelWriter(f"{OUT}/experiment_results.xlsx") as xw:
    class_table.to_excel(xw, sheet_name="Dataset")
    summary.T.to_excel(xw, sheet_name="Summary_mean_CI")
    mean_t.to_excel(xw, sheet_name="Means")
    ci_t.to_excel(xw, sheet_name="CI95_halfwidth")
    stats_table.to_excel(xw, sheet_name="Statistical_tests", index=False)
    stability.to_excel(xw, sheet_name="Stability", index=False)
    freq.to_frame().to_excel(xw, sheet_name="Feature_frequency")
    per_class_mean.to_excel(xw, sheet_name="Per_class_F1")
    results.to_excel(xw, sheet_name="All_runs", index=False)
    selected.assign(features=selected["features"].apply(", ".join)).to_excel(xw, sheet_name="Selected_features", index=False)
    pd.DataFrame({"n_features": pareto_example[:, 1] * len(FEATURES), "val_macro_F1": 1 - pareto_example[:, 0],
                  "F1_drop_noise": pareto_example[:, 2]}).to_excel(xw, sheet_name="Pareto_front_run1", index=False)
    pd.DataFrame(cm_run1, index=CLASSES, columns=CLASSES).to_excel(xw, sheet_name="Confusion_matrix_run1")

pd.set_option("display.width", 250)
text = "\n".join([
    "REVISED EXPERIMENT SUMMARY — CICIoMT2024", "=" * 70,
    f"Runs: {results['run'].nunique()} | train rows/class: {TRAIN_PER_CLASS} | test rows/class: {TEST_PER_CLASS}",
    f"NSGA-II: pop={GA_POP}, gen={GA_GEN}, rows/class={GA_PER_CLASS}, {GA_FOLDS}-fold CV, {GA_TREES} trees, "
    f"tolerance={F1_TOLERANCE}, consensus={CONSENSUS}",
    f"Noise: relative error on real-valued features, levels={NOISE_LEVELS}, draws={NOISE_DRAWS}, main={NOISE_MAIN}",
    f"Final model: class-balanced Random Forest, {FINAL_TREES} trees | features for selection (IAT excluded): {len(FEATURES)}",
    "\nDataset:\n" + class_table.to_string(),
    "\nMAIN RESULTS (mean ± 95% CI):\n" + summary.T.to_string(),
    "\nSTATISTICAL TESTS:\n" + stats_table.round(5).to_string(index=False),
    "\nSTABILITY:\n" + stability.round(4).to_string(index=False),
    "\nFEATURE SELECTION FREQUENCY (proposed):\n" + freq[freq > 0].round(2).to_string(),
    "\nPER-CLASS F1 (mean):\n" + per_class_mean.round(4).to_string(),
])
with open(f"{OUT}/summary_for_manuscript.txt", "w", encoding="utf-8") as fh:
    fh.write(text)
print(text)
print("\nSaved:", sorted(os.listdir(OUT)))

## Step 13 — Figures

All results are already saved. Each figure has its own cell; if one fails, nothing is lost.

In [ ]:
SHORT = {"All features + IAT (reference)": "All+IAT\n(ref.)", "All features": "All", "MI (top-k)": "MI",
         "RFE (top-k)": "RFE", "NSGA-II (2-obj)": "NSGA-II\n2-obj", "NSGA-II (3-obj)": "NSGA-II\n3-obj",
         PROPOSED: "Proposed"}
COLOR = {"All features + IAT (reference)": "#D9D9D9", "All features": "#7F7F7F", "MI (top-k)": "#5B9BD5",
         "RFE (top-k)": "#ED7D31", "NSGA-II (2-obj)": "#A5A5A5", "NSGA-II (3-obj)": "#70AD47", PROPOSED: "#1F4E78"}
labels = [SHORT[m] for m in METHOD_ORDER]; colors = [COLOR[m] for m in METHOD_ORDER]

In [ ]:
# Figure 1: macro-F1, clean vs 10% noise
fig, ax = plt.subplots(figsize=(8, 3.8)); x = np.arange(len(METHOD_ORDER)); w = 0.38
ax.bar(x - w/2, mean_t["macro_F1"], w, yerr=ci_t["macro_F1"], label="Clean test", color="#1F4E78", capsize=3)
ax.bar(x + w/2, mean_t["F1_noise_10"], w, yerr=ci_t["F1_noise_10"], label="10% measurement noise", color="#ED7D31", capsize=3)
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=8); ax.set_ylabel("Macro-F1")
ax.set_ylim(max(0, mean_t[["macro_F1", "F1_noise_10"]].values.min() - 0.08), 1.0); ax.legend(fontsize=8)
ax.set_title("Figure 1. Detection performance, clean vs noisy (mean ± 95% CI)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig1_f1_clean_vs_noisy.png"); plt.show(); plt.close("all")

In [ ]:
# Figure 2: robustness curve, macro-F1 against noise level
fig, ax = plt.subplots(figsize=(6.5, 4))
lv = [0] + [int(l * 100) for l in NOISE_LEVELS]
for m in METHOD_ORDER:
    y = [mean_t.loc[m, "macro_F1"]] + [mean_t.loc[m, c] for c in NOISE_COLS]
    ax.plot(lv, y, "o-", color=COLOR[m], lw=2.5 if m == PROPOSED else 1.3, label=SHORT[m].replace("\n", " "))
ax.set_xlabel("Measurement noise on real-valued features (%)"); ax.set_ylabel("Macro-F1"); ax.set_xticks(lv)
ax.legend(fontsize=7); ax.set_title("Figure 2. Robustness to measurement noise")
plt.tight_layout(); plt.savefig(f"{OUT}/fig2_robustness_curve.png"); plt.show(); plt.close("all")

In [ ]:
# Figure 3: accuracy vs number of features
fig, ax = plt.subplots(figsize=(6.5, 4))
for m in METHOD_ORDER:
    ax.errorbar(mean_t.loc[m, "n_features"], mean_t.loc[m, "macro_F1"], xerr=ci_t.loc[m, "n_features"],
                yerr=ci_t.loc[m, "macro_F1"], fmt="o", color=COLOR[m], capsize=3, markersize=8,
                label=SHORT[m].replace("\n", " "))
ax.set_xlabel("Number of features"); ax.set_ylabel("Macro-F1"); ax.legend(fontsize=7)
ax.set_title("Figure 3. Accuracy vs. number of features")
plt.tight_layout(); plt.savefig(f"{OUT}/fig3_tradeoff.png"); plt.show(); plt.close("all")

In [ ]:
# Figure 4: computational cost
fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
for ax, met, lab in zip(axes, ["train_time_s", "infer_ms_per_1000", "model_MB_est"],
                        ["Training time (s)", "Inference (ms / 1,000 records)", "Model size (MB, est.)"]):
    ax.bar(labels, mean_t[met], yerr=ci_t[met], color=colors, capsize=3)
    ax.set_ylabel(lab); ax.tick_params(axis="x", labelsize=6)
fig.suptitle("Figure 4. Computational cost (mean ± 95% CI)"); plt.tight_layout()
plt.savefig(f"{OUT}/fig4_cost.png"); plt.show(); plt.close("all")

In [ ]:
# Figure 5: Pareto front of the 3-objective search (run 1)
F = pareto_example
fig, ax = plt.subplots(figsize=(6.5, 4))
sc = ax.scatter(F[:, 1] * len(FEATURES), 1 - F[:, 0], c=F[:, 2], cmap="viridis_r", s=50, edgecolor="k")
plt.colorbar(sc, label="F1 drop under 10% noise (lower = more robust)")
ax.set_xlabel("Number of features"); ax.set_ylabel("Cross-validated macro-F1 (training data)")
ax.set_title("Figure 5. NSGA-II Pareto front (run 1)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig5_pareto_front.png"); plt.show(); plt.close("all")

In [ ]:
# Figure 6: how often each feature is selected by the proposed method
top = freq[freq > 0]
fig, ax = plt.subplots(figsize=(6.5, max(3, 0.25 * len(top))))
ax.barh(top.index[::-1], top.values[::-1], color="#1F4E78")
ax.set_xlabel("Share of runs in which the feature was selected"); ax.set_xlim(0, 1)
ax.set_title("Figure 6. Features chosen by the proposed method")
plt.tight_layout(); plt.savefig(f"{OUT}/fig6_feature_frequency.png"); plt.show(); plt.close("all")

In [ ]:
# Figure 7: confusion matrix of the proposed method (run 1), row-normalised
fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(cm_run1, cmap="Blues", vmin=0, vmax=1); ax.grid(False)
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES, fontsize=8)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, f"{cm_run1[i, j]:.2f}", ha="center", va="center", fontsize=7,
                color="white" if cm_run1[i, j] > 0.5 else "black")
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); plt.colorbar(im, ax=ax)
ax.set_title("Figure 7. Confusion matrix, proposed (run 1)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig7_confusion_matrix.png"); plt.show(); plt.close("all")